# Reference Dataset vs Generated — Marginal & Stylized-Facts Analysis

Compares **generated Close log-return paths** (from `generate_samples.py` CSV output)
against a **reference distribution built from the entire `SP500WindowDataset`**
(all sliding windows, L=512, stride=100, over the raw CSV files).

Unlike `OHLC_conditional_csv_analysis.ipynb`, the reference here is **not** limited to
the handful of windows that were sampled — it covers the complete window corpus,
giving a truer population baseline.

Plots produced:
1. Aggregate statistics table (normalized + unnormalized)
2. Marginal distribution (histogram / QQ / ECDF)
3. Thinner marginal distribution (central 96 %)
4. Increments distribution (first differences)
5. Stylized-facts overlap (`sf.distribution`, `sf.acf`, `sf.leverage_effect`)
6. Price sample paths at aggregate level

In [27]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
CHECKPOINT_STEM = (
    "csv20000_samples5_steps200_seed50_20260604_233534_UNCO_Pm-1.4_Ps1.8"
)  # sub-folder under GEN_DIR produced by generate_samples.py

GEN_DIR  = "../data/generated/final/edm"   # base output directory of generate_samples.py
SPLIT    = "both"   # "train" | "val" | "both"

# Reference dataset — the full sliding-window corpus
REFERENCE_ROOT_DIR = "../data/SNP500_individual_normalized_replication"
SEQ_LEN  = 512
STRIDE   = 100
COLUMNS  = ("date", "close", "open", "high", "low")  # must match training config
CLOSE_FEATURE_IDX = 0   # index of Close in the feature dimension (K axis)

# Normalization stats — used only for unnormalized plots
STATS_FILE = "../data/general/normalization_stats_norm_replication.csv"

# Image output
OUTPUT_DIR = "../images/final/edm/reference_vs_generated"
appendix   = "ref_full_corpus_UNCO_Pm-1.4_Ps1.8"  # string appended to output filenames to distinguish checkpoints
# ─────────────────────────────────────────────────────────────────────────────

## 0. Imports

In [28]:
import os

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf as sm_acf
from pathlib import Path
import warnings

sys.path.insert(0, "..")
import torch
from torch.utils.data import DataLoader
from src.utils.dataloader import SP500WindowDataset, csdi_collate_fn
import replication.stylized_facts as sf

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

Output directory: c:\Users\Lenovo\Documents\SCUOLA\UNI\MASTER\2ANNO\THESIS\Master-Thesis\images\final\edm\reference_vs_generated


## 1. Load generated CSV data

Reads the `*_generated_close.csv` files written by `generate_samples.py`.
Each row is one *(window, sample)* pair; step columns hold the L=512 time steps.
Shapes after parsing:
- `gen_3d`: `(n_gen_windows, n_samples, seq_len)` — generated Close
- `gt_close`: `(n_gen_windows, seq_len)` — ground-truth Close stored alongside (for reference only)

In [29]:
def load_split(split, ckpt_dir):
    gen_path  = os.path.join(ckpt_dir, f"{split}_generated_close.csv")
    ohlc_path = os.path.join(ckpt_dir, f"{split}_gt_ohlc.csv")
    assert os.path.isfile(gen_path),  f"Not found: {gen_path}"
    assert os.path.isfile(ohlc_path), f"Not found: {ohlc_path}"
    df_gen  = pd.read_csv(gen_path)
    df_ohlc = pd.read_csv(ohlc_path)
    step_cols = [c for c in df_gen.columns if c.startswith("step_")]
    print(f"[{split}] windows={df_gen['window_idx'].nunique()}  "
          f"samples={df_gen['sample_idx'].nunique()}  seq_len={len(step_cols)}")
    return df_gen, df_ohlc, step_cols


def parse_arrays(df_gen, df_ohlc, step_cols):
    window_ids = sorted(df_gen["window_idx"].unique())
    n_windows  = len(window_ids)
    n_samples  = df_gen["sample_idx"].nunique()
    seq_len    = len(step_cols)
    gen_3d = np.zeros((n_windows, n_samples, seq_len), dtype=np.float32)
    for wi, wid in enumerate(window_ids):
        sub = df_gen[df_gen["window_idx"] == wid].sort_values("sample_idx")
        gen_3d[wi] = sub[step_cols].to_numpy(dtype=np.float32)
    def gt_feat(feat_name):
        sub = df_ohlc[df_ohlc["feature"] == feat_name].sort_values("window_idx")
        return sub[step_cols].to_numpy(dtype=np.float32)
    return gen_3d, gt_feat("close"), gt_feat("open"), gt_feat("high"), gt_feat("low"), window_ids


ckpt_dir = os.path.join(GEN_DIR, CHECKPOINT_STEM)
assert os.path.isdir(ckpt_dir), f"Directory not found: {ckpt_dir}"

splits_to_run = ["train", "val"] if SPLIT == "both" else [SPLIT]
parsed_gen = {}
for split in splits_to_run:
    try:
        df_gen, df_ohlc, step_cols = load_split(split, ckpt_dir)
        parsed_gen[split] = parse_arrays(df_gen, df_ohlc, step_cols)
    except AssertionError as e:
        print(f"Skipping {split}: {e}")

# Pool all available splits into one flat set for distribution-level comparison
gen_3d_list, gt_close_list = [], []
for g3d, gtc, *_ in parsed_gen.values():
    gen_3d_list.append(g3d)
    gt_close_list.append(gtc)

gen_3d_all  = np.concatenate(gen_3d_list,  axis=0)   # (N_gen, n_samples, L)
gt_close_all = np.concatenate(gt_close_list, axis=0)  # (N_gen, L)

n_gen_windows, n_samples, seq_len = gen_3d_all.shape
steps = np.arange(seq_len)
print(f"\nPooled generated: gen_3d_all={gen_3d_all.shape}  gt_close_all={gt_close_all.shape}")

[train] windows=20000  samples=5  seq_len=512
[val] windows=237  samples=5  seq_len=512

Pooled generated: gen_3d_all=(20237, 5, 512)  gt_close_all=(20237, 512)


## 2. Build the reference window corpus

Instantiates `SP500WindowDataset` with the same `seq_len` and `stride` used during training,
then iterates through **all** windows and collects the Close log-return sequences.

This gives a reference distribution that covers the complete dataset — not just the
handful of windows stored in the generated CSVs.

Shape: `ref_close` → `(n_ref_windows, seq_len)`

In [30]:
print("Building SP500WindowDataset …")
ref_dataset = SP500WindowDataset(
    root_dir=REFERENCE_ROOT_DIR,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    columns=COLUMNS,
    time_mode="global_index",
    cache_data=True,        # load each file once into RAM
    drop_incomplete=True,
)
print(f"  {len(ref_dataset):,} windows across {len(ref_dataset.files)} files")

ref_loader = DataLoader(
    ref_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,          # single-threaded for notebook stability
    collate_fn=csdi_collate_fn,
    drop_last=False,
)

close_chunks = []
for batch in ref_loader:
    # observed_data: (B, K, L)  — K features in order matching COLUMNS[1:]
    obs = batch["observed_data"]                          # (B, K, L)
    close_chunks.append(obs[:, CLOSE_FEATURE_IDX, :].numpy())  # (B, L)

ref_close = np.concatenate(close_chunks, axis=0)          # (n_ref_windows, L)
print(f"Reference corpus: ref_close={ref_close.shape}  "
      f"total scalars={ref_close.size:,}")

Building SP500WindowDataset …
  25,226 windows across 210 files
Reference corpus: ref_close=(25226, 512)  total scalars=12,915,712


## 3. Normalization stats and unnormalization helper

Loads the per-feature mean and standard deviation used during dataset normalisation.
The helper `unnorm(arr, feature)` inverts the z-score transform:
$x_{\text{raw}} = x_{\text{norm}} \cdot \sigma + \mu$

In [31]:
stats_df = pd.read_csv(STATS_FILE, header=0, index_col=0)
stats_df.columns = ["mean", "std"]
stats_df.index   = stats_df.index.str.strip().str.lower()
col_stats = stats_df.to_dict(orient="index")

print("Normalization statistics:")
display(stats_df)

def unnorm(arr, feature):
    """Invert z-score normalisation for the given feature."""
    mu  = col_stats[feature]["mean"]
    sig = col_stats[feature]["std"]
    return arr * sig + mu

# Unnormalized arrays (log-returns in original scale)
gen_3d_unnorm  = unnorm(gen_3d_all, "close")    # (N_gen, n_samples, L)
ref_close_unnorm = unnorm(ref_close, "close")    # (n_ref_windows, L)

print(f"\ngen_3d_unnorm  : {gen_3d_unnorm.shape}")
print(f"ref_close_unnorm: {ref_close_unnorm.shape}")

Normalization statistics:


,mean,std
close,0.000451,0.020867
open,0.000265,0.010859
high,0.012179,0.019033
low,-0.011306,0.017881



gen_3d_unnorm  : (20237, 5, 512)
ref_close_unnorm: (25226, 512)


## 4. Flat views for distribution-level comparisons

Collapse window/sample dimensions into 1-D arrays for scalar-level statistics.

| Array | Content | Shape |
|---|---|---|
| `gen_flat` | all generated Close values (normalized) | `(N_gen × n_samples × L,)` |
| `ref_flat` | all reference Close values (normalized) | `(n_ref_windows × L,)` |
| `gen_flat_un` | same, unnormalized | |
| `ref_flat_un` | same, unnormalized | |

In [32]:
gen_flat    = gen_3d_all.ravel()          # normalized
ref_flat    = ref_close.ravel()           # normalized
gen_flat_un = gen_3d_unnorm.ravel()       # unnormalized
ref_flat_un = ref_close_unnorm.ravel()    # unnormalized

print(f"gen_flat : {gen_flat.shape}   ref_flat : {ref_flat.shape}")
print(f"ratio gen/ref scalars : {len(gen_flat)/len(ref_flat):.2f}x")

gen_flat : (51806720,)   ref_flat : (12915712,)
ratio gen/ref scalars : 4.01x


## 5. Aggregate statistics

Summary statistics (mean, std, quantiles, skewness, excess kurtosis) computed
both in **normalized** and **original (unnormalized)** space.

Fat tails should produce excess kurtosis >> 0.
A well-calibrated model should match the reference across all rows.

In [33]:
def summary_stats(x, label):
    return {
        "source"          : label,
        "n"               : len(x),
        "mean"            : np.mean(x),
        "std"             : np.std(x),
        "min"             : np.min(x),
        "q01"             : np.quantile(x, 0.01),
        "q05"             : np.quantile(x, 0.05),
        "median"          : np.median(x),
        "q95"             : np.quantile(x, 0.95),
        "q99"             : np.quantile(x, 0.99),
        "max"             : np.max(x),
        "skewness"        : float(scipy_stats.skew(x)),
        "excess_kurtosis" : float(scipy_stats.kurtosis(x, fisher=True)),
    }


print("=" * 70)
print("  AGGREGATE STATISTICS — NORMALIZED SPACE")
print("=" * 70)
df_norm = pd.DataFrame([
    summary_stats(gen_flat, "Generated Close (norm)"),
    summary_stats(ref_flat, "Reference Close (norm)"),
]).set_index("source")
display(df_norm.T.round(6))

print()
print("=" * 70)
print("  AGGREGATE STATISTICS — UNNORMALIZED LOG-RETURNS")
print("=" * 70)
df_unnorm = pd.DataFrame([
    summary_stats(gen_flat_un, "Generated Close (unnorm)"),
    summary_stats(ref_flat_un, "Reference Close (unnorm)"),
]).set_index("source")
display(df_unnorm.T.round(4))

  AGGREGATE STATISTICS — NORMALIZED SPACE


source,Generated Close (norm),Reference Close (norm)
n,5.180672e+07,1.291571e+07
mean,7.840000e-04,1.710000e-04
std,7.523000e-01,9.996960e-01
min,-6.023737e+01,-4.489045e+01
q01,-2.029233e+00,-2.717974e+00
q05,-1.117401e+00,-1.420743e+00
median,-2.143400e-02,-2.163300e-02
q95,1.132108e+00,1.438079e+00
q99,2.085504e+00,2.774559e+00
max,8.222971e+01,3.319643e+01



  AGGREGATE STATISTICS — UNNORMALIZED LOG-RETURNS


source,Generated Close (unnorm),Reference Close (unnorm)
n,5.180672e+07,1.291571e+07
mean,5.000000e-04,5.000000e-04
std,1.570000e-02,2.090000e-02
min,-1.256500e+00,-9.363000e-01
q01,-4.190000e-02,-5.630000e-02
q05,-2.290000e-02,-2.920000e-02
median,0.000000e+00,0.000000e+00
q95,2.410000e-02,3.050000e-02
q99,4.400000e-02,5.830000e-02
max,1.716300e+00,6.931000e-01


## 6. Marginal distribution

Three views on the scalar marginal of Close log-returns (normalized space):

- **Histogram** (density) — overall shape and tail alignment
- **QQ plot** — quantile-level agreement; departure from the diagonal reveals tail over/under-generation
- **ECDF** — smooth comparison; the KS statistic is the maximum vertical gap

In [34]:
ks_stat, ks_p = scipy_stats.ks_2samp(gen_flat, ref_flat)
w1_dist       = scipy_stats.wasserstein_distance(gen_flat, ref_flat)
print(f"KS statistic = {ks_stat:.4f}  |  p-value = {ks_p:.4e}")
print(f"Wasserstein-1 distance = {w1_dist:.6f}")

lo, hi   = np.quantile(ref_flat, [0.001, 0.999])
bins     = np.linspace(lo, hi, 80)
probs    = np.linspace(0.001, 0.999, 5_000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(ref_flat, bins=bins, density=True, alpha=0.5, label="Reference",  color="tab:blue")
ax.hist(gen_flat, bins=bins, density=True, alpha=0.5, label="Generated",  color="tab:orange")
ax.set_title("Marginal distribution of Close (normalized)")
ax.set_xlabel("normalized log-return"); ax.set_ylabel("density"); ax.legend(fontsize=8)

ax = axes[1]
q_ref = np.quantile(ref_flat, probs)
q_gen = np.quantile(gen_flat, probs)
ax.scatter(q_ref, q_gen, s=4, alpha=0.4, color="steelblue")
lims = [min(q_ref.min(), q_gen.min()), max(q_ref.max(), q_gen.max())]
ax.plot(lims, lims, "r--", lw=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — Generated vs Reference")
ax.set_xlabel("Reference quantiles"); ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

ax = axes[2]
for vals, label, color in [(ref_flat, "Reference", "tab:blue"), (gen_flat, "Generated", "tab:orange")]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, lw=1.2)
ax.set_xlim(lo, hi)
ax.set_title(f"ECDF  (KS={ks_stat:.3f}, p={ks_p:.2e})")
ax.set_xlabel("normalized log-return"); ax.set_ylabel("CDF"); ax.legend(fontsize=8)

plt.suptitle("Marginal distribution — Generated vs Full Reference Corpus", y=1.02)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"marginal_distribution_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

KeyboardInterrupt: 

## 7. Thinner marginal distribution (central 96 %)

Repeats the marginal plot trimmed to the [2 %, 98 %] quantile range of the
combined distribution. This zooms in on the body of the distribution
and avoids extreme tail points dominating the histogram bins.

The trimmed KS statistic tests agreement specifically in the central region.

In [ ]:
lo2, hi2 = np.quantile(ref_flat, [0.02, 0.98])
bins2    = np.linspace(lo2, hi2, 80)
probs2   = np.linspace(0.02, 0.98, 5_000)

# Trimmed KS (central 98 % of combined support)
cq01, cq99 = np.quantile(np.concatenate([gen_flat, ref_flat]), [0.01, 0.99])
gen_trim = gen_flat[(gen_flat >= cq01) & (gen_flat <= cq99)]
ref_trim = ref_flat[(ref_flat >= cq01) & (ref_flat <= cq99)]
ks_trim, ks_trim_p = scipy_stats.ks_2samp(gen_trim, ref_trim, method="asymp")
print(f"Trimmed KS (central 98 %) = {ks_trim:.4f}  |  p = {ks_trim_p:.4e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(ref_flat, bins=bins2, density=True, alpha=0.5, label="Reference",  color="tab:blue")
ax.hist(gen_flat, bins=bins2, density=True, alpha=0.5, label="Generated",  color="tab:orange")
ax.set_title("Marginal (central 96 %)")
ax.set_xlabel("normalized log-return"); ax.set_ylabel("density"); ax.legend(fontsize=8)

ax = axes[1]
q_ref2 = np.quantile(ref_flat, probs2)
q_gen2 = np.quantile(gen_flat, probs2)
ax.scatter(q_ref2, q_gen2, s=4, alpha=0.4, color="steelblue")
lims2 = [min(q_ref2.min(), q_gen2.min()), max(q_ref2.max(), q_gen2.max())]
ax.plot(lims2, lims2, "r--", lw=1.2, label="y = x (perfect)")
ax.set_title("QQ plot (central 96 %) — Generated vs Reference")
ax.set_xlabel("Reference quantiles"); ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

ax = axes[2]
for vals, label, color in [(ref_flat, "Reference", "tab:blue"), (gen_flat, "Generated", "tab:orange")]:
    s = np.sort(vals)
    mask = (s >= lo2) & (s <= hi2)
    ax.plot(s[mask], np.arange(1, len(s) + 1)[mask] / len(s), label=label, lw=1.2)
ax.set_xlim(lo2, hi2)
ax.set_title(f"ECDF trimmed  (KS trim={ks_trim:.3f})")
ax.set_xlabel("normalized log-return"); ax.set_ylabel("CDF"); ax.legend(fontsize=8)

plt.suptitle("Thinner marginal distribution (central 96 %) — Generated vs Reference", y=1.02)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"marginal_thinner_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

Trimmed KS (central 98 %) = 0.0420  |  p = 0.0000e+00


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10848\752967811.py:40: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


<Figure size 1500x400 with 3 Axes>

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10848\752967811.py:42: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(os.path.join(OUTPUT_DIR, f"marginal_thinner_{appendix}.png"), dpi=300, bbox_inches="tight")


## 8. Increments distribution

First differences $\Delta_t = C_t - C_{t-1}$ within each window, flattened across
all windows (and samples for generated paths).

Increments capture the step-to-step dynamics. A good model should match:
- the scale (std of increments ≈ reference)
- tail shape (excess kurtosis of increments)
- the QQ of the increment distribution

In [ ]:
# Compute increments — shape collapses windows × samples
gen_paths_2d = gen_3d_all.reshape(-1, seq_len)  # (N_gen × n_samples, L)
gen_inc = np.diff(gen_paths_2d, axis=1).ravel()
ref_inc = np.diff(ref_close, axis=1).ravel()

ks_inc, ks_p_inc = scipy_stats.ks_2samp(gen_inc, ref_inc)
w1_inc           = scipy_stats.wasserstein_distance(gen_inc, ref_inc)
print(f"Increment KS = {ks_inc:.4f}  |  p = {ks_p_inc:.4e}")
print(f"Wasserstein-1 (increments) = {w1_inc:.6f}")

df_inc = pd.DataFrame([
    summary_stats(gen_inc, "Generated ΔClose"),
    summary_stats(ref_inc, "Reference ΔClose"),
]).set_index("source")
display(df_inc.T.round(6))

lo_i, hi_i = np.quantile(ref_inc, [0.001, 0.999])
bins_i  = np.linspace(lo_i, hi_i, 80)
probs_i = np.linspace(0.001, 0.999, 5_000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(ref_inc, bins=bins_i, density=True, alpha=0.5, label="Reference ΔClose",  color="tab:blue")
ax.hist(gen_inc, bins=bins_i, density=True, alpha=0.5, label="Generated ΔClose",  color="tab:orange")
ax.set_title(f"Increment marginal  (KS={ks_inc:.3f}, p={ks_p_inc:.2e})")
ax.set_xlabel("ΔClose (normalized)"); ax.legend(fontsize=8)

ax = axes[1]
q_ref_i = np.quantile(ref_inc, probs_i)
q_gen_i = np.quantile(gen_inc, probs_i)
ax.scatter(q_ref_i, q_gen_i, s=4, alpha=0.4, color="steelblue")
lims_i = [min(q_ref_i.min(), q_gen_i.min()), max(q_ref_i.max(), q_gen_i.max())]
ax.plot(lims_i, lims_i, "r--", lw=1.2)
ax.set_title("QQ plot — Generated ΔClose vs Reference")
ax.set_xlabel("Reference quantiles"); ax.set_ylabel("Generated quantiles")

# ACF of increments averaged over a subset of paths
N_ACF_PATHS = 500
LAGS        = min(30, seq_len // 2)
lag_axis    = np.arange(1, LAGS + 1)

def mean_acf_paths(paths_2d, lags, n_max=N_ACF_PATHS, use_diff=True):
    idx = np.random.default_rng(0).choice(len(paths_2d), size=min(n_max, len(paths_2d)), replace=False)
    acfs = []
    for i in idx:
        x = np.diff(paths_2d[i]) if use_diff else paths_2d[i]
        acfs.append(sm_acf(x, nlags=lags, fft=True)[1:])
    return np.mean(acfs, axis=0)

ax = axes[2]
ax.plot(lag_axis, mean_acf_paths(ref_close,    LAGS), label="Reference ΔClose ACF",  color="tab:blue",   lw=1.4)
ax.plot(lag_axis, mean_acf_paths(gen_paths_2d, LAGS), label="Generated ΔClose ACF",  color="tab:orange", lw=1.4)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("ACF of Close increments (mean over paths)")
ax.set_xlabel("lag"); ax.set_ylabel("autocorrelation"); ax.legend(fontsize=8)

plt.suptitle("Increments distribution — Generated vs Full Reference Corpus", y=1.02)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"increments_distribution_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

Increment KS = 0.0196  |  p = 0.0000e+00
Wasserstein-1 (increments) = 0.040704


source,Generated ΔClose,Reference ΔClose
n,5.171064e+07,1.289049e+07
mean,6.400000e-05,7.100000e-05
std,1.346378e+00,1.422083e+00
min,-6.693008e+01,-6.643613e+01
q01,-3.645329e+00,-3.856991e+00
q05,-1.935686e+00,-2.026469e+00
median,-1.144000e-03,-9.550000e-04
q95,1.949710e+00,2.039293e+00
q99,3.718110e+00,3.924426e+00
max,5.075181e+01,4.931985e+01


<Figure size 1500x400 with 3 Axes>

## 9. Stylized facts — individual plots

Uses `replication.stylized_facts` to compute:
- **`sf.distribution`** — log-log density plot of the return distribution (heavy tails)
- **`sf.acf`** — autocorrelation of absolute returns (volatility clustering)
- **`sf.leverage_effect`** — cross-correlation between past returns and future volatility

Each function writes its own plot to disk; we save to `OUTPUT_DIR`.

> `sf.distribution` and `sf.acf` expect a **1-D flat array** and an **object array of paths** respectively.

In [ ]:
sf_output_dir = Path(OUTPUT_DIR) / "stylized_facts"
sf_output_dir.mkdir(parents=True, exist_ok=True)

# ── object arrays of 1-D paths (required by sf.acf and sf.leverage_effect) ───
# For the reference we use all windows (one path per window).
# For generated we flatten window × sample so every sample is its own path.
ref_paths_obj = np.empty(len(ref_close), dtype=object)
for i, p in enumerate(ref_close):
    ref_paths_obj[i] = p

gen_paths_obj = np.empty(len(gen_paths_2d), dtype=object)
for i, p in enumerate(gen_paths_2d):
    gen_paths_obj[i] = p

print(f"ref_paths_obj : {ref_paths_obj.shape}  path_len={ref_paths_obj[0].shape}")
print(f"gen_paths_obj : {gen_paths_obj.shape}  path_len={gen_paths_obj[0].shape}")

ref_paths_obj : (25226,)  path_len=(512,)
gen_paths_obj : (101195,)  path_len=(512,)


In [ ]:
# sf.distribution — log-log density of scalar return values
ref_dist_x, ref_dist_y = sf.distribution(
    ref_flat,
    file_name=str(sf_output_dir / f"ref_distribution_{appendix}"),
    scale="log", multiple=False, normalize=True, granuality=100,
)

gen_dist_x, gen_dist_y = sf.distribution(
    gen_flat,
    file_name=str(sf_output_dir / f"gen_distribution_{appendix}"),
    scale="log", multiple=False, normalize=True, granuality=100,
)

In [ ]:
# subsample generated paths to match reference size (or any fixed N)
subsample = True
if subsample:
    rng = np.random.default_rng(42)
    n_sub = int(len(ref_paths_obj) * 0.5)
    idx_gen = rng.choice(len(gen_paths_obj), size=n_sub, replace=False)
    idx_ref = rng.choice(len(ref_paths_obj), size=n_sub, replace=False)
    gen_paths_obj_sub = gen_paths_obj[idx_gen]
    ref_paths_obj_sub = ref_paths_obj[idx_ref]


In [ ]:
# sf.acf — autocorrelation of absolute returns (volatility clustering)
MAX_LAG_SF = min(1000, seq_len // 2)

ref_acf_res = sf.acf(
    ref_paths_obj,
    file_name=str(sf_output_dir / f"ref_acf_{appendix}"),
    for_abs=True, multiple=True, fit=False, scale="log", max_lag=MAX_LAG_SF,
)

if not subsample:
    ref_acf_res = sf.acf(
        ref_paths_obj,
        file_name=str(sf_output_dir / f"ref_acf_{appendix}"),
        for_abs=True, multiple=True, fit=False, scale="log", max_lag=MAX_LAG_SF,
    )
    gen_acf_res = sf.acf(
        gen_paths_obj,
        file_name=str(sf_output_dir / f"gen_acf_{appendix}"),
        for_abs=True, multiple=True, fit=False, scale="log", max_lag=MAX_LAG_SF,
    )
else:
    ref_acf_res = sf.acf(
        ref_paths_obj_sub,
        file_name=str(sf_output_dir / f"ref_acf_subsampled_{appendix}"),
        for_abs=True, multiple=True, fit=False, scale="log", max_lag=MAX_LAG_SF,
    )
    gen_acf_res = sf.acf(
        gen_paths_obj_sub,
        file_name=str(sf_output_dir / f"gen_acf_subsampled_{appendix}"),
        for_abs=True, multiple=True, fit=False, scale="log", max_lag=MAX_LAG_SF,
    )

In [ ]:
# sf.leverage_effect — cross-correlation past return / future absolute return
MAX_LEV_LAG = min(100, seq_len // 2)

ref_levs = sf.leverage_effect(
    ref_paths_obj,
    file_name=str(sf_output_dir / f"ref_leverage_{appendix}"),
    multiple=True, min_lag=1, max_lag=MAX_LEV_LAG,
)

gen_levs = sf.leverage_effect(
    gen_paths_obj,
    file_name=str(sf_output_dir / f"gen_leverage_{appendix}"),
    multiple=True, min_lag=1, max_lag=MAX_LEV_LAG,
)

## 10. Stylized facts — overlap plots

Draws Reference and Generated on the same axes for direct comparison.
Blue = reference corpus; orange = generated samples.

### 10a. Return distribution (log-log)

In [ ]:
for sign, mask_fn, suffix in [
    ( 1, lambda x: x > 0, "pos"),
    (-1, lambda x: x < 0, "neg"),
]:
    m_ref = mask_fn(ref_dist_x)
    m_gen = mask_fn(gen_dist_x)

    fig = plt.figure(dpi=150)
    plt.plot(sign * ref_dist_x[m_ref], ref_dist_y[m_ref], ".", color="tab:blue",   label="Reference")
    plt.plot(sign * gen_dist_x[m_gen], gen_dist_y[m_gen], ".", color="tab:orange", label="Generated")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel(r"Normalized scale in $\sigma$", fontsize=14)
    plt.ylabel(r"Probability Density $P(r)$",  fontsize=14)
    plt.title(f"Return distribution ({'positive' if sign == 1 else 'negative'} tail)")
    plt.legend(fontsize=12)
    plt.tight_layout()
    display(fig)
    plt.savefig(
        str(sf_output_dir / f"overlap_distribution_{suffix}_{appendix}.png"),
        transparent=True, dpi=300,
    )
    plt.close(fig)

<Figure size 960x720 with 1 Axes>

<Figure size 960x720 with 1 Axes>

### 10b. Volatility clustering (ACF of |returns|, log-log)

In [ ]:
lags_sf = np.linspace(1, ref_acf_res.size, ref_acf_res.size)

fig = plt.figure(dpi=150)
plt.plot(lags_sf, ref_acf_res, ".", color="tab:blue",   label="Reference")
plt.plot(lags_sf, gen_acf_res, ".", color="tab:orange", label="Generated")
plt.ylim(1e-5, 1.0)
plt.xscale("log"); plt.yscale("log")
plt.xlabel(r"Lag $k$", fontsize=14)
plt.ylabel("Autocorrelation of |returns|", fontsize=14)
plt.title("Volatility clustering")
plt.legend(fontsize=12)
plt.tight_layout()
display(fig)
plt.savefig(
    str(sf_output_dir / f"overlap_acf_{appendix}.png"),
    transparent=True, dpi=300,
)
plt.close(fig)

<Figure size 960x720 with 1 Axes>

### 10c. Leverage effect

In [ ]:
lag_range = range(1, MAX_LEV_LAG)

fig = plt.figure(dpi=150)
plt.plot(lag_range, ref_levs, color="tab:blue",   label="Reference")
plt.plot(lag_range, gen_levs, color="tab:orange", label="Generated")
plt.axhline(0, color="black", lw=0.8, ls="--")
plt.xlabel(r"$t$", fontsize=14)
plt.ylabel(r"$L(t)$", fontsize=14)
plt.title("Leverage effect")
plt.legend(fontsize=12)
plt.tight_layout()
display(fig)
plt.savefig(
    str(sf_output_dir / f"overlap_leverage_{appendix}.png"),
    transparent=True, dpi=300,
)
plt.close(fig)

<Figure size 960x720 with 1 Axes>

## 11. Price sample paths — aggregate level

Converts **unnormalized** Close log-returns to price paths with $S_0 = 1$
via cumulative product of $\exp(r_t)$.

A random subset of reference windows and generated paths are overlaid on a single
panel **without any per-window conditioning information** — the goal is a visual
check of the aggregate behaviour (drift, scale of fluctuations, path diversity).

In [ ]:
N_PATHS_SHOW = 150    # number of paths shown per group
SEED_SHOW    = 15

rng_show = np.random.default_rng(SEED_SHOW)

# Reference: pick N random windows and convert to price paths
ref_idx   = rng_show.choice(len(ref_close_unnorm), size=min(N_PATHS_SHOW, len(ref_close_unnorm)), replace=False)
ref_price = np.cumprod(np.exp(ref_close_unnorm[ref_idx]), axis=1)  # (N, L)

# Generated: reshape to (N_gen × n_samples, L), pick N random paths
gen_unnorm_2d = gen_3d_unnorm.reshape(-1, seq_len)                 # (N_gen*n_samples, L)
gen_idx       = rng_show.choice(len(gen_unnorm_2d), size=min(N_PATHS_SHOW, len(gen_unnorm_2d)), replace=False)
gen_price     = np.cumprod(np.exp(gen_unnorm_2d[gen_idx]), axis=1) # (N, L)

cmap   = plt.cm.tab20
n_ref  = len(ref_price)
n_gen  = len(gen_price)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

ax = axes[0]
for i, p in enumerate(ref_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
ax.axhline(1.0, color="black", lw=0.7, ls=":", label="$S_0 = 1$")
ax.set_title(f"Reference paths (n={n_ref})")
ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
ax.legend(fontsize=9)

ax = axes[1]
for i, p in enumerate(gen_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
ax.axhline(1.0, color="black", lw=0.7, ls=":", label="$S_0 = 1$")
ax.set_title(f"Generated paths (n={n_gen})")
ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
ax.legend(fontsize=9)

plt.suptitle(
    f"Price sample paths — aggregate level  (S₀ = 1, unnormalized log-returns → prices)",
    y=1.01, fontsize=13,
)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"price_paths_aggregate_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)


<Figure size 1500x500 with 2 Axes>

## 12. Price paths — side-by-side overlay

Both groups plotted on a **single panel** for direct scale comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

n_ref = len(ref_price)
n_gen = len(gen_price)
ref_colors = plt.cm.Blues(np.linspace(0.35, 0.9, n_ref))
gen_colors = plt.cm.Oranges(np.linspace(0.35, 0.9, n_gen))

for p, c in zip(ref_price, ref_colors):
    ax.plot(steps, p, color=c, lw=0.5, alpha=0.4)
for p, c in zip(gen_price, gen_colors):
    ax.plot(steps, p, color=c, lw=0.5, alpha=0.4)

# Legend proxies
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="tab:blue",   lw=1.5, label=f"Reference (n={n_ref})"),
    Line2D([0], [0], color="tab:orange", lw=1.5, label=f"Generated (n={n_gen})"),
]
ax.legend(handles=legend_elements, fontsize=11)
ax.axhline(1.0, color="black", lw=0.7, ls=":")
ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"price_paths_overlay_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

<Figure size 1200x500 with 1 Axes>

# Unnormalized statistics

In [36]:
# ── Wasserstein-1 & KS test on unnormalized log-returns ──────────────────────
rng = np.random.default_rng(42)
gen_sub = rng.choice(gen_flat, size=len(ref_flat), replace=False)
ks_stat, ks_p = scipy_stats.ks_2samp(gen_sub, ref_flat)
w1_un = scipy_stats.wasserstein_distance(gen_sub, ref_flat)
print(f"KS statistic SUBSAMPLED = {ks_stat:.4f}  |  p-value = {ks_p:.4e}")
print(f"Wasserstein-1 distance SUBSAMPLED = {w1_un:.6f}")

ks_un, ks_p_un = scipy_stats.ks_2samp(gen_flat_un, ref_flat_un)
w1_un          = scipy_stats.wasserstein_distance(gen_flat_un, ref_flat_un)
print(f"KS statistic (unnorm) = {ks_un:.4f}  |  p-value = {ks_p_un:.4e}")
print(f"Wasserstein-1 (unnorm) = {w1_un:.8f}")

# Increments in unnormalized space
gen_inc_un = np.diff(gen_3d_unnorm.reshape(-1, seq_len), axis=1).ravel()
ref_inc_un = np.diff(ref_close_unnorm, axis=1).ravel()

ks_inc_un, ks_p_inc_un = scipy_stats.ks_2samp(gen_inc_un, ref_inc_un)
w1_inc_un              = scipy_stats.wasserstein_distance(gen_inc_un, ref_inc_un)
print(f"\nKS statistic (unnorm increments) = {ks_inc_un:.4f}  |  p-value = {ks_p_inc_un:.4e}")
print(f"Wasserstein-1 (unnorm increments) = {w1_inc_un:.8f}")


KS statistic SUBSAMPLED = 0.0442  |  p-value = 0.0000e+00
Wasserstein-1 distance SUBSAMPLED = 0.134499
KS statistic (unnorm) = 0.0442  |  p-value = 0.0000e+00
Wasserstein-1 (unnorm) = 0.00280979

KS statistic (unnorm increments) = 0.0390  |  p-value = 0.0000e+00
Wasserstein-1 (unnorm increments) = 0.00397399
